# Batch SSI Null Builds To `data/processed`

This notebook runs the SSI/null place-field pipeline for paired sessions in `data/interim` and saves one processed folder per session:

`data/processed/<mouse>/ssi/<session_id>/`

Each completed folder contains:

- `pfnull.npz`
- `ssi_classification.csv`
- `run_config.json`

The notebook previews the session list by default. Set `RUN_NOW = True` in the config cell to launch the full batch.

In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

REPO = Path.cwd().resolve()
if not (REPO / 'src' / 'placefields').exists():
    REPO = Path(r'C:/Users/tadse/OneDrive/Documenti/GitHub/CA1-cellclass').resolve()
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

from placefields.interim_io import find_pairs

INTERIM_ROOT = REPO / 'data' / 'interim'
PROCESSED_ROOT = REPO / 'data' / 'processed'
WORK_ROOT = PROCESSED_ROOT / '_ssi_build_work'
CELL_CLASSIFICATION_TABLE = (
    REPO / 'results' / 'type_u_comparison_valero_feats_3' / 'cell_classification_table.csv'
)

PYTHON_EXE = sys.executable
SCRIPT = REPO / 'scripts' / 'build_placefield_null_from_interim.py'
if not SCRIPT.exists():
    SCRIPT = REPO / 'scripts' / 'pipelines' / 'build_placefield_null_from_interim.py'

# Keep these defaults aligned with single_session_null_ssi.ipynb.
NULL_METHOD = 'circular_shift'   # 'random' | 'poisson' | 'circular_shift'
NB_REP = 1000
XBIN_REM = 0
MIN_SPEED = 2.0
SMOOTH_SIGMA_BINS = 2.0
SM_ALPHA = 0.05
SAVE_SSI_NULL = True

# Empty list means all paired sessions in INTERIM_ROOT.
SELECTED_SESSIONS: list[str] = []

# Safety switches.
RUN_NOW = True
OVERWRITE = False
CLEAN_TEMP_WORKDIR = True

print(f'Repo: {REPO}')
print(f'Interim root: {INTERIM_ROOT}')
print(f'Processed root: {PROCESSED_ROOT}')
print(f'Pipeline script: {SCRIPT}')
print(f'Python: {PYTHON_EXE}')
print(f'Cell classification table exists: {CELL_CLASSIFICATION_TABLE.exists()}')

Repo: C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass
Interim root: C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\data\interim
Processed root: C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\data\processed
Pipeline script: C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\scripts\build_placefield_null_from_interim.py
Python: c:\Users\tadse\miniconda3\envs\odors\python.exe
Cell classification table exists: True


## Discover Sessions

In [2]:
session_filter = set(SELECTED_SESSIONS)
pairs = find_pairs(INTERIM_ROOT, session_filter)

records = []
for session_id, allcel_path, traj_path in pairs:
    rel_parent = allcel_path.parent.relative_to(INTERIM_ROOT)
    mouse = rel_parent.parts[0] if rel_parent.parts else session_id.split('_', 1)[0]
    out_dir = PROCESSED_ROOT / mouse / 'ssi' / session_id
    records.append(
        {
            'mouse': mouse,
            'session_id': session_id,
            'allcel_npz': allcel_path,
            'traj_npz': traj_path,
            'out_dir': out_dir,
            'pfnull_exists': (out_dir / 'pfnull.npz').exists(),
            'classification_exists': (out_dir / 'ssi_classification.csv').exists(),
            'run_config_exists': (out_dir / 'run_config.json').exists(),
        }
    )

sessions_df = pd.DataFrame(records)
if not sessions_df.empty:
    sessions_df = sessions_df.sort_values(['mouse', 'session_id'], kind='stable').reset_index(drop=True)

print(f'Paired sessions found: {len(sessions_df)}')
display(sessions_df.head(30))

Paired sessions found: 66


,mouse,session_id,allcel_npz,traj_npz,out_dir,pfnull_exists,classification_exists,run_config_exists
0,VS100,VS100_2024-11-03_14-40-11,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
1,VS100,VS100_2024-11-04_15-23-28,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
2,VS101,VS101_2024-11-04_16-46-04,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
3,VS101,VS101_2024-11-05_17-44-14,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
4,VS102,VS102_2024-11-05_18-59-31,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
5,VS102,VS102_2024-11-07_16-01-28,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
6,VS103,VS103_2024-11-10_14-58-13,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
7,VS104,VS104_2024-11-10_16-39-46,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
8,VS106,VS106_2024-11-12_14-55-21,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False
9,VS109,VS109_2024-12-20_17-20-40,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False,False,False


## Batch Helpers

In [3]:
def final_outputs_exist(out_dir: Path) -> bool:
    return all(
        (out_dir / name).exists()
        for name in ('pfnull.npz', 'ssi_classification.csv', 'run_config.json')
    )


def build_command(row: pd.Series) -> tuple[Path, list[str]]:
    session_id = str(row['session_id'])
    work_dir = WORK_ROOT / session_id
    cmd = [
        str(PYTHON_EXE),
        str(SCRIPT),
        '--interim_root',
        str(INTERIM_ROOT),
        '--out_root',
        str(work_dir),
        '--sessions',
        session_id,
        '--null_method',
        str(NULL_METHOD),
        '--nb_rep',
        str(int(NB_REP)),
        '--xbin_rem',
        str(int(XBIN_REM)),
        '--min_speed',
        str(float(MIN_SPEED)),
        '--smooth_sigma_bins',
        str(float(SMOOTH_SIGMA_BINS)),
        '--sm_alpha',
        str(float(SM_ALPHA)),
        '--classification_csv',
        'ssi_classification.csv',
        '--cell_classification_table',
        str(CELL_CLASSIFICATION_TABLE),
        '--no_run_subdir',
    ]
    if SAVE_SSI_NULL:
        cmd.append('--save_ssi_null')
    if OVERWRITE:
        cmd.append('--overwrite')
    return work_dir, cmd


def write_processed_run_config(
    *,
    src_config: Path,
    dst_config: Path,
    row: pd.Series,
    command: list[str],
    work_dir: Path,
    pfnull_source: Path,
) -> None:
    if src_config.exists():
        cfg = json.loads(src_config.read_text(encoding='utf-8'))
    else:
        cfg = {}
    cfg['processed_ssi_layout'] = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'session_id': str(row['session_id']),
        'mouse': str(row['mouse']),
        'processed_session_dir': str(row['out_dir']),
        'pfnull_npz': str(dst_config.parent / 'pfnull.npz'),
        'ssi_classification_csv': str(dst_config.parent / 'ssi_classification.csv'),
        'pipeline_work_dir': str(work_dir),
        'pipeline_pfnull_source': str(pfnull_source),
        'command': command,
    }
    dst_config.write_text(json.dumps(cfg, indent=2), encoding='utf-8')


def pipeline_index_summary(work_dir: Path, session_id: str) -> dict[str, object]:
    index_path = work_dir / 'run_index.csv'
    if not index_path.exists():
        return {}
    try:
        index_df = pd.read_csv(index_path)
        match = index_df[index_df['session_id'].astype(str) == str(session_id)]
        summary: dict[str, object] = {'pipeline_run_index': str(index_path)}
        if match.empty:
            return summary
        pipeline_row = match.iloc[-1].to_dict()
        for key in ('status', 'error', 'pfnull_npz', 'allcel_npz', 'traj_npz'):
            if key in pipeline_row:
                summary[f'pipeline_{key}'] = pipeline_row[key]
        return summary
    except Exception as exc:
        return {
            'pipeline_run_index': str(index_path),
            'pipeline_index_error': str(exc),
        }


def collect_outputs(row: pd.Series, work_dir: Path, command: list[str]) -> dict[str, object]:
    session_id = str(row['session_id'])
    out_dir = Path(row['out_dir'])
    pfnull_candidates = sorted(work_dir.rglob(f'{session_id}_pfnull.npz'))
    if not pfnull_candidates:
        raise FileNotFoundError(f'No pfnull output found under {work_dir}')
    pfnull_src = pfnull_candidates[0]

    class_src = work_dir / 'ssi_classification.csv'
    config_src = work_dir / 'run_config.json'
    if not class_src.exists():
        raise FileNotFoundError(f'Missing classification CSV: {class_src}')
    if not config_src.exists():
        raise FileNotFoundError(f'Missing run config: {config_src}')

    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pfnull_src, out_dir / 'pfnull.npz')
    shutil.copy2(class_src, out_dir / 'ssi_classification.csv')
    write_processed_run_config(
        src_config=config_src,
        dst_config=out_dir / 'run_config.json',
        row=row,
        command=command,
        work_dir=work_dir,
        pfnull_source=pfnull_src,
    )

    if CLEAN_TEMP_WORKDIR and work_dir.exists():
        shutil.rmtree(work_dir)

    return {
        'pfnull_npz': str(out_dir / 'pfnull.npz'),
        'ssi_classification_csv': str(out_dir / 'ssi_classification.csv'),
        'run_config_json': str(out_dir / 'run_config.json'),
    }


def run_one_session(row: pd.Series) -> dict[str, object]:
    session_id = str(row['session_id'])
    out_dir = Path(row['out_dir'])
    if final_outputs_exist(out_dir) and not OVERWRITE:
        return {
            'session_id': session_id,
            'mouse': str(row['mouse']),
            'status': 'skip_exists',
            'out_dir': str(out_dir),
        }

    work_dir, cmd = build_command(row)
    if OVERWRITE and work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    proc = subprocess.run(
        cmd,
        cwd=str(REPO),
        capture_output=True,
        text=True,
        check=False,
    )
    if proc.returncode != 0:
        return {
            'session_id': session_id,
            'mouse': str(row['mouse']),
            'status': 'error',
            'returncode': int(proc.returncode),
            'out_dir': str(out_dir),
            'stdout_tail': proc.stdout[-2000:],
            'stderr_tail': proc.stderr[-2000:],
        }

    try:
        copied = collect_outputs(row, work_dir, cmd)
    except Exception as exc:
        return {
            'session_id': session_id,
            'mouse': str(row['mouse']),
            'status': 'error_no_output',
            'returncode': int(proc.returncode),
            'out_dir': str(out_dir),
            'work_dir': str(work_dir),
            'error': str(exc),
            'stdout_tail': proc.stdout[-2000:],
            'stderr_tail': proc.stderr[-2000:],
            **pipeline_index_summary(work_dir, session_id),
        }
    return {
        'session_id': session_id,
        'mouse': str(row['mouse']),
        'status': 'ok',
        'returncode': int(proc.returncode),
        'out_dir': str(out_dir),
        **copied,
    }

## Preview

In [4]:
if sessions_df.empty:
    raise RuntimeError(f'No paired allcel/trajdata sessions found under {INTERIM_ROOT}')

preview_df = sessions_df[['mouse', 'session_id', 'out_dir']].copy()
preview_df['complete'] = preview_df['out_dir'].map(lambda path: final_outputs_exist(Path(path)))
display(preview_df.head(50))

work_dir0, cmd0 = build_command(sessions_df.iloc[0])
print('Example work dir:')
print(work_dir0)
print('\nExample command:')
print(subprocess.list2cmdline(cmd0))

,mouse,session_id,out_dir,complete
0,VS100,VS100_2024-11-03_14-40-11,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
1,VS100,VS100_2024-11-04_15-23-28,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
2,VS101,VS101_2024-11-04_16-46-04,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
3,VS101,VS101_2024-11-05_17-44-14,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
4,VS102,VS102_2024-11-05_18-59-31,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
5,VS102,VS102_2024-11-07_16-01-28,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
6,VS103,VS103_2024-11-10_14-58-13,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
7,VS104,VS104_2024-11-10_16-39-46,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
8,VS106,VS106_2024-11-12_14-55-21,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False
9,VS109,VS109_2024-12-20_17-20-40,C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-c...,False


Example work dir:
C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\data\processed\_ssi_build_work\VS100_2024-11-03_14-40-11

Example command:
c:\Users\tadse\miniconda3\envs\odors\python.exe C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\scripts\build_placefield_null_from_interim.py --interim_root C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\data\interim --out_root C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\data\processed\_ssi_build_work\VS100_2024-11-03_14-40-11 --sessions VS100_2024-11-03_14-40-11 --null_method circular_shift --nb_rep 1000 --xbin_rem 0 --min_speed 2.0 --smooth_sigma_bins 2.0 --sm_alpha 0.05 --classification_csv ssi_classification.csv --cell_classification_table C:\Users\tadse\OneDrive\Documenti\GitHub\CA1-cellclass\results\type_u_comparison_valero_feats_3\cell_classification_table.csv --no_run_subdir --save_ssi_null


## Run Batch

In [5]:
if not RUN_NOW:
    print('Preview only. Set RUN_NOW = True in the config cell to run the batch.')
else:
    results: list[dict[str, object]] = []
    n_sessions = len(sessions_df)
    for i, (_, row) in enumerate(sessions_df.iterrows(), start=1):
        result = run_one_session(row)
        results.append(result)
        print(f"[{i}/{n_sessions}] {result['status']}: {result['session_id']}")

    results_df = pd.DataFrame(results)
    batch_index = PROCESSED_ROOT / 'ssi_batch_run_index.csv'
    batch_index.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(batch_index, index=False)
    print(f'Wrote batch index: {batch_index}')
    display(results_df)

Preview only. Set RUN_NOW = True in the config cell to run the batch.


## Inspect One Output

In [6]:
completed = sessions_df[
    sessions_df['out_dir'].map(lambda path: final_outputs_exist(Path(path)))
]
if completed.empty:
    print('No completed processed SSI folders yet.')
else:
    sample_dir = Path(completed.iloc[0]['out_dir'])
    print(f'Sample processed SSI folder: {sample_dir}')
    print(sorted(path.name for path in sample_dir.iterdir()))
    display(pd.read_csv(sample_dir / 'ssi_classification.csv').head())

No completed processed SSI folders yet.
